[DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) (C) Fabrizio Smeraldi, 2020,2024 ([f.smeraldi@qmul.ac.uk](mailto:f.smeraldi@qmul.ac.uk) - [web](http://www.eecs.qmul.ac.uk/~fabri/)). This notebook is released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

# COVID-19 Hospital Admissions | England
Interactive UKHSA disease tracking dashboard. Explore daily admissions, smoothed trends and historical periods.

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests

plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})

## Data source
Historical data load from `admissions.json` so the dashboard works even when the UKHSA API is unavailable. The **Fetch Data** button retrieves newer observations when possible.

In [2]:
DATA_FILE = Path('admissions.json')
if not DATA_FILE.exists():
    raise FileNotFoundError('Place admissions.json in the same folder as Dashboard.ipynb')
with DATA_FILE.open(encoding='utf-8') as f:
    jsondata = json.load(f)

def wrangle_data(rawdata):
    if isinstance(rawdata, dict):
        rawdata = rawdata.get('results', [])
    data = pd.DataFrame(rawdata)
    if data.empty or not {'date','metric_value'}.issubset(data.columns):
        raise ValueError('No usable date/metric_value observations returned')
    data['date'] = pd.to_datetime(data['date'], errors='coerce')
    data['metric_value'] = pd.to_numeric(data['metric_value'], errors='coerce')
    data = data.dropna(subset=['date','metric_value'])
    return (data.groupby('date')['metric_value'].mean()
            .sort_index().to_frame('admissions'))

df = wrangle_data(jsondata)
print(f'Loaded {len(df):,} daily observations, {df.index.min():%d %b %Y} – {df.index.max():%d %b %Y}')

Loaded 2,250 daily observations, 01 Aug 2020 – 30 Sep 2026


## Explore admissions
Use the year selector and smoothing option to compare the COVID-19 hospital admission waves. **All years** shows the entire dataset; a 7-day average reduces daily noise.

In [3]:
year_options = [('All years', 'all')] + [(str(y),y) for y in sorted(df.index.year.unique())]
year = widgets.Dropdown(options=year_options, description='Year:', value='all')
view = widgets.ToggleButtons(options=[('Daily','daily'),('7-day average','average'),('Both','both')], value='both', description='Display:')
scale = widgets.Checkbox(value=False, description='Logarithmic scale')
chart_output = widgets.Output()
stats_output = widgets.Output()

def draw_chart(*_):
    with chart_output:
        clear_output(wait=True)
        subset = df if year.value == 'all' else df.loc[df.index.year == year.value]
        if subset.empty:
            print('No observations for this selection')
            return
        avg = subset['admissions'].rolling(7, min_periods=1).mean()
        fig, ax = plt.subplots(figsize=(11,4.5))
        if view.value in ('daily','both'):
            ax.plot(subset.index,subset['admissions'],alpha=.55,lw=1,label='Daily admissions')
        if view.value in ('average','both'):
            ax.plot(subset.index,avg,lw=2.2,label='7-day rolling average')
        if scale.value:
            ax.set_yscale('log')
        ax.set(title='COVID-19 hospital admissions in England', ylabel='Admissions per day',xlabel='Date')
        ax.legend(loc='upper right')
        ax.grid(alpha=.2)
        fig.autofmt_xdate()
        plt.show()
    with stats_output:
        clear_output(wait=True)
        print(f'Days: {len(subset):,}  |  Average daily admissions: {subset.admissions.mean():,.0f}  |  Peak: {subset.admissions.max():,.0f} on {subset.admissions.idxmax():%d %b %Y}')

for control in (year,view,scale):
    control.observe(draw_chart,names='value')
display(widgets.VBox([widgets.HBox([year,view]),scale,stats_output,chart_output]))
draw_chart()

## Refresh from UKHSA
Fetch Data requests the most recent available admission records and merges them into the saved historical series. If the API is offline or returns no data, the current chart remains unchanged.

In [4]:
API_URL = ('https://api.ukhsa-dashboard.data.gov.uk/themes/infectious_disease/'
           'sub_themes/respiratory/topics/COVID-19/geography_types/Nation/'
           'geographies/England/metrics/COVID-19_healthcare_admissionByDay')

def access_api():
    response = requests.get(API_URL, params={'page_size':365,'page':1}, timeout=20,
                            headers={'Accept':'application/json'})
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict):
        return payload.get('results',[])
    return payload

fetch_button = widgets.Button(description='Fetch Data', icon='refresh', button_style='info')
fetch_status = widgets.HTML(value='Using locally saved UKHSA data.')

def api_button_callback(button):
    global df, jsondata
    button.disabled=True
    fetch_status.value='Fetching latest UKHSA observations…'
    try:
        incoming = access_api()
        newdf = wrangle_data(incoming)
        merged = pd.concat([df,newdf]).groupby(level=0).last().sort_index()
        df=merged
        # Keep the existing historical dataset plus updated records
        combined = {record['date']:record for record in jsondata if 'date' in record}
        for record in incoming:
            if 'date' in record:
                combined[record['date']]=record
        jsondata=list(combined.values())
        year.options=[('All years','all')]+[(str(y),y) for y in sorted(df.index.year.unique())]
        draw_chart()
        fetch_status.value=f'Updated successfully: {len(newdf)} records received. Latest date: {df.index.max():%d %b %Y}.'
        button.icon='check'
    except Exception as exc:
        fetch_status.value=f'Could not refresh: {type(exc).__name__}: {exc}. Existing data retained.'
        button.icon='warning'
    finally:
        button.disabled=False

fetch_button.on_click(api_button_callback)
display(widgets.HBox([fetch_button,fetch_status]))

## Interpretation and limitations
The plot shows **daily hospital admissions associated with COVID-19 in England**, not all COVID-19 infections. A 7-day moving average highlights broad waves while smoothing reporting fluctuations. Reporting delays and changes in testing, admission practices and surveillance can affect comparisons. The data are a historical snapshot unless refreshed; the dashboard does not automatically update its bundled JSON file.

**Source and attribution:** Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/) published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency) and on the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) by Fabrizio Smeraldi. Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).